# Simple EDA — German grid load and residual load

**Spec:** [`specs/01-Simple-EDA.md`](../specs/01-Simple-EDA.md) ·
**Data:** `data/smard.csv` (SMARD / Bundesnetzagentur, hourly, region DE)

## What this notebook is for

Understand the SMARD dataset well enough to make informed modeling decisions for the
1-day-ahead **residual load** forecast, and describe — descriptively, without defining
thresholds — where the extreme residual load cases that motivate the project actually sit.

It answers five questions:

1. Is the dataset complete, correctly typed, and continuous enough to be treated as an hourly
   time series?
2. Are our aggregation helpers correct, and what exactly do our calendar conventions mean?
3. What are the trend, seasonal and calendar structures in each series?
4. How do the series relate to each other, and which relations are candidate features?
5. What does the `residual_load` distribution look like, and how do its tails behave in time?

## How to read it

- **§6 is a contract, not a private choice.** The week convention, the season definition, the
  reporting units and the descriptive-slice policy fixed there are inherited by
  [`specs/02-Deep-EDA.md`](../specs/02-Deep-EDA.md) rather than re-derived.
- **Every plot** carries a title, axis labels and explicit units — MWh, average MW or MWh/day,
  never an unlabelled number — and is followed by one to three sentences saying what it shows.
- **No thresholds.** This notebook does not define a risk flag, a cut-off or a labelled column.
  Where it shows "the tail hours" it selects them by rank, for description only (§6.5).
- **Units.** Values are energy per hourly interval in MWh. Over an hourly interval that number
  is also the average power in MW, which is why the hourly mean of a series and its average MW
  are the same number — §6.4 makes that explicit rather than leaving it to be inferred.

---

## 1 · Setup

`data/` is **gitignored**, so `data/smard.csv` does not come with a clone. Regenerate it by
running [`notebooks/API-connection.ipynb`](API-connection.ipynb) top to bottom — it pulls the
SMARD API (no key required) and writes the file in German Excel CSV format
(`sep=";"`, `decimal=","`, `utf-8-sig`).

In [ ]:
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["figure.max_open_warning"] = 0  # this notebook draws ~30 figures on purpose

# Works whether the kernel starts in notebooks/ (Jupyter) or at the repo root (nbconvert).
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "smard.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running notebooks/API-connection.ipynb top to bottom."
    )

print(f"pandas {pd.__version__} · numpy {np.__version__} · seaborn {sns.__version__}")
print(f"reading {DATA}")

---

## 2 · Helpers

`period_mean`, `style_timeseries` and `seasonal_plot` are **copied** from
[`notebooks/EDA-robert.ipynb`](EDA-robert.ipynb), which this spec does not modify. Three
changes apply to the copies here:

1. `ylabel` becomes a **required** argument in `style_timeseries` and `seasonal_plot`. The
   originals default it to `"(MWh)"`, which silently violates the reporting convention of §6.5.
2. `seasonal_plot` **honours** `ylabel`. The original accepts it, documents it, and then
   hard-codes `ax.set_ylabel("MWh")`.
3. `period_mean`'s edge rule moves into `_complete_periods` so it exists in exactly one place,
   shared with the new `period_energy`. The arithmetic is unchanged — §6.3 proves it.

`period_mean`'s docstring also drops the phrase *"the first and last one"*. The rule is **drop
periods the data does not fully cover**, which is not the same thing: this record starts exactly
on a month boundary, so January 2022 is complete and only the trailing month is dropped. The
code was always right; the prose was not.

In [ ]:
def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    The one edge rule of this notebook, in one place. A period counts only if it starts no
    earlier than the first observation and ends no later than the last observation's closing
    edge. Both `period_mean` and `period_energy` defer to this.
    """
    periods = index.to_period(freq).unique().sort_values()
    complete = (periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + pd.Timedelta("1h")
    )
    return periods[complete]


def period_mean(time_series, freq):
    """Mean of `time_series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts. Note the rule is "not fully covered", not "the first and the last":
    see the correctness test in section 6.3.

    Copied from notebooks/EDA-robert.ipynb; the edge rule was factored into `_complete_periods`
    without changing the result.
    """
    agg = time_series.groupby(time_series.index.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(time_series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def period_energy(time_series, freq, drop_incomplete=True):
    """Per-period aggregate of `time_series` in both project reporting units.

    Returns a DataFrame indexed by period start:

    ``mwh_per_day``
        period sum / calendar days in the period — the energy view (MWh/day).
    ``avg_mw``
        period sum / hours **actually present** — the level view (MW). Deliberately not
        ``mwh_per_day / 24``: a month containing the spring DST switch holds 743 hours, not 744.
    ``hours``, ``days``
        the two denominators, exposed so section 6.4's comparison table needs no second copy of
        this arithmetic.

    Incomplete periods are dropped by the same `_complete_periods` rule as `period_mean`. Pass
    ``drop_incomplete=False`` to keep them — only the 6.4 table does, because it has to *show*
    the period the rule discards.
    """
    grouped = time_series.groupby(time_series.index.to_period(freq))
    total, hours = grouped.sum(), grouped.size()
    periods = total.index

    if drop_incomplete:
        keep = _complete_periods(time_series.index, freq)
        total, hours, periods = total.loc[keep], hours.loc[keep], keep

    # Freq-generic: 7 for every week, 28-31 for months. `days_in_month` would be "M"-only.
    days = (periods.end_time.normalize() - periods.start_time).days + 1

    # .to_numpy() on every right-hand side: aligning a PeriodIndex-backed Series against a
    # DatetimeIndex-derived array silently yields all-NaN.
    return pd.DataFrame(
        {
            "mwh_per_day": total.to_numpy() / days,
            "avg_mw": total.to_numpy() / hours.to_numpy(),
            "hours": hours.to_numpy(),
            "days": np.asarray(days),
        },
        index=periods.start_time,
    )

In [ ]:
def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required (not defaulted to "(MWh)" as in the original): every plot must state
    whether it shows MWh, average MW or MWh/day. See the reporting convention in section 6.5.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")


def seasonal_plot(df, y_value, title, ylabel):
    """Creates a seasonal plot from a dataframe.

    Args:
        df (DataFrame): frame with separate `month` and `year` columns, already aggregated to
            one row per (year, month). Passing raw hourly data makes seaborn bootstrap a
            confidence interval per cell over 41k rows — minutes of runtime, meaningless band.
        y_value (str): name of the y-value to plot
        title (str): title of the plot
        ylabel (str): axis description, including units. Required, and actually applied — the
            original hard-coded "MWh" and ignored this argument.
    """
    fig, ax = plt.subplots(figsize=(14, 5))

    sns.lineplot(
        data=df,
        x="month",
        y=y_value,
        hue="year",
        palette="viridis",
        legend=True,
        ax=ax
    )
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.show()

---

## 3 · Load and prepare

The CSV is German Excel format, so every numeric column arrives as text with a comma decimal
separator. The failure mode to guard against is silent: unconverted columns land as a string
dtype, every aggregate still computes something, and every number is wrong. The dtype assertion
below is the guard.

The flat, `RangeIndex`ed frame is called `raw` and is **deleted** at the end of the loading
cells. Everything downstream uses `ts`, so the two cannot drift apart.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb writes them. Note the inconsistent
# capitalisation in the source ("Grid Load" vs "Forecast Grid load") — reproduced deliberately.
COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid load": "fc_grid_load",
    "Forecast Residual Load": "fc_res",
}

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

assert set(raw.columns) == {"timestamp"} | set(COLUMNS), (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ ({'timestamp'} | set(COLUMNS)))}"
)

print("as read from disk — note the comma decimals and the string dtypes:")
display(raw.head(3))
display(raw.dtypes.to_frame("dtype"))

In [ ]:
raw = raw.rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

ts = raw.set_index("timestamp").sort_index()
del raw  # the flat frame does not outlive the loading cells

ts.head(3)

In [ ]:
print(f"shape           : {ts.shape[0]:,} rows x {ts.shape[1]} columns")
print(f"index           : {ts.index.min()}  ->  {ts.index.max()}")
print(f"index monotonic : {ts.index.is_monotonic_increasing}, unique: {ts.index.is_unique}")
display(ts.dtypes.to_frame("dtype"))

# Positive is_float_dtype test, not `!= object`: under pandas 3 an unconverted German-decimal
# column lands as StringDtype, and `!= object` would wave it straight through.
assert all(pd.api.types.is_float_dtype(ts[c]) for c in COLUMNS.values()), ts.dtypes

# Snapshot for the self-check in section 11: re-asserted at the end, so a cell inserted anywhere
# in between that mutates `ts` is caught regardless of which vintage of the CSV was loaded.
LOADED = {"rows": len(ts), "start": ts.index.min(), "end": ts.index.max()}

display(ts.describe().T)

41 107 hourly rows spanning 2022-01-01 00:00 to 2026-09-09 23:00, all eight series `float64`,
nothing obviously degenerate in `describe()`. The one number worth pausing on is
`residual_load`'s minimum: it is **negative**, which is valid data (renewable oversupply), not
an error. That rules out log scales and log transforms for this series throughout.

---

## 4 · Derived columns and the `SERIES` constant

All derived columns are defined **here, in one place**, immediately after loading. Two of them
are needed by the data quality audit itself (`renewables` for the identity check, `hour` for the
night-solar check), so they cannot wait for the section that first plots them.

`SERIES` names the eight data columns. Without it, `ts.corr()` in §9 would silently pull the
derived columns into the correlation heatmap, and `describe()` would report on `year` and `dow`
as though they were measurements.

> **On "nine series".** The spec says nine; the CSV has eight numeric columns. Its Data table
> has nine rows only because it counts `timestamp`. Eight it is — `renewables` is derived, and
> appears in §9.2 where its relation to `residual_load` is the actual subject rather than a
> tautology cluttering a heatmap.

`season`/`season_year` and `spans_gap` are created here with everything else; the *reasoning*
behind them lives where it is used — the gap narrative in §5.2, the season contract in §6.5.

In [ ]:
SERIES = [
    "wind_off", "wind_on", "solar", "grid_load", "residual_load",
    "fc_gen_wind_solar", "fc_grid_load", "fc_res",
]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter. Rationale in 6.5.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

ts["renewables"] = ts[["wind_on", "wind_off", "solar"]].sum(axis=1)
ts["year"] = ts.index.year
ts["month"] = ts.index.month
ts["hour"] = ts.index.hour
ts["dow"] = ts.index.dayofweek
ts["is_weekend"] = ts.index.dayofweek >= 5
ts["date"] = ts.index.date
ts["season"] = pd.Categorical(
    ts.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
ts["season_year"] = ts.index.year + (ts.index.month == 12)
# True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
ts["spans_gap"] = ts.index.to_series().diff() > pd.Timedelta("1h")

DERIVED = [
    "renewables", "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(ts.columns) == SERIES + DERIVED, list(ts.columns)
print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {ts.shape[1]} columns")
ts[DERIVED].head(3)

`ts` now carries exactly the eight data columns plus ten declared derived ones, and the assertion
above is what keeps that true. The closing self-check in §11 re-runs it against the same two
lists — which is the mechanical proof that no flag or label column crept in along the way.

One consequence to keep in mind for the rest of the notebook: `date` is object dtype, so a bare
`ts.groupby(...).mean()` now raises under pandas 3. Every aggregation from here on names its
columns explicitly.

---

## 5 · Data quality audit

Six checks: coverage, gaps, duplicates, missing values, value ranges, and the residual load
identity. The governing rule for this whole section is that **nothing is repaired**. Gaps are
found, named and documented; they are not filled, interpolated or reindexed away. How to handle
them is a modeling decision, and it belongs to the modeling spec, not to an audit.

### 5.1 · Coverage

In [ ]:
full_index = pd.date_range(ts.index.min(), ts.index.max(), freq="h")

print(f"first timestamp          : {ts.index.min()}")
print(f"last timestamp           : {ts.index.max()}")
print(f"rows in file             : {len(ts):,}")
print(f"complete hourly index    : {len(full_index):,}")
print(f"difference               : {len(full_index) - len(ts):,} hours missing")

The file is **5 hours short** of a complete hourly index over its own span: 41 107 rows where
41 112 would be needed. That is small enough to be invisible in any aggregate and large enough to
break anything that assumes a fixed row-to-hour mapping. §5.2 identifies every one of them.

### 5.2 · Gaps, and the two different ways this record loses hours

In [ ]:
missing_hours = full_index.difference(ts.index)
N_GAPS = len(missing_hours)

display(
    pd.DataFrame(
        {"weekday": missing_hours.day_name(), "hour_label": missing_hours.hour},
        index=missing_hours,
    ).rename_axis("missing timestamp")
)

# `spans_gap` marks the row FOLLOWING each gap; the two views must agree.
assert (ts.index[ts["spans_gap"]] == missing_hours + pd.Timedelta("1h")).all()
print(f"{N_GAPS} missing hours, and spans_gap flags exactly the {ts['spans_gap'].sum()} rows after them\n")

print("the 2025 spring switch, hour by hour -- 01:00 is followed directly by 03:00:")
display(ts.loc["2025-03-30 00:00":"2025-03-30 04:00", ["grid_load", "residual_load", "spans_gap"]])

All five missing timestamps are a **Sunday at 02:00 in late March** — the spring DST switch. At
02:00 CET the clock jumps to 03:00 CEST, so the wall-clock hour labelled 02:00 does not exist on
those days and the file correctly has no row for it. This is not missing data; it is a missing
*hour*.

The second, less obvious loss is the **autumn DST fold**, and it is a genuinely different effect
that no gap search can find:

In [ ]:
autumn_switches = pd.DatetimeIndex(
    [
        pd.date_range(f"{y}-10-01", f"{y}-10-31", freq="W-SUN")[-1]
        for y in range(ts.index.year.min(), ts.index.year.max() + 1)
    ]
)
autumn_switches = autumn_switches[autumn_switches <= ts.index.max()]

days = ts.index.normalize()
display(
    pd.DataFrame(
        {
            "rows_in_file": [(days == d).sum() for d in autumn_switches],
            "rows_labelled_02h": [((days == d) & (ts.index.hour == 2)).sum() for d in autumn_switches],
            "true_local_hours": 25,
        },
        index=autumn_switches.date,
    ).rename_axis("autumn switch date")
)

Each autumn switch day really has **25** local hours — 02:00 occurs twice, once at CEST and once
at CET — but the file carries **24 rows with a single 02:00 row**. SMARD has already collapsed
the repeated hour rather than emitting it twice, so one physical hour per autumn switch is merged
away.

The consequence matters for the rest of the notebook: the fold produces **neither an index gap
nor a duplicate timestamp**, so `missing_hours` and the duplicate check in §5.3 are both blind to
it. Four autumn switches sit in this record against five spring ones, because the data ends on
2026-09-09, before that year's October switch.

**Nothing above is repaired.** No `fillna`, no `interpolate`, no `reindex` appears anywhere in
this notebook. What the gaps mean for `.diff()`, `.shift()`, rolling windows and the ACF is
flagged at each site where it bites.

### 5.3 · Duplicate timestamps

In [ ]:
print(f"duplicate timestamps: {ts.index.duplicated().sum()}")

Zero, as expected — and the reason is precisely the collapse described in §5.2. A dataset that
emitted the autumn fold honestly *would* show one duplicate per autumn switch. So this zero is
evidence that SMARD pre-processed the fold, **not** evidence that the record is hour-complete.
Read together with §5.1, the two results say: no hour appears twice, and five hours are absent.

### 5.4 · Missing values

In [ ]:
display(
    pd.DataFrame(
        {
            "n_missing": ts[SERIES].isna().sum(),
            "share_%": (ts[SERIES].isna().mean() * 100).round(3),
        }
    )
)

**Zero missing values in all eight series.** An explicit zero is a finding, not an absence of
one: it means no imputation strategy is needed for the columns themselves, and every NaN that
turns up later in the notebook is one *we* created — by a `.shift()`, a lag or an incomplete
aggregation period — rather than one that arrived with the data.

### 5.5 · Value ranges

A naive range check drowns in false positives, so the night-solar test carries an explicit
tolerance: night-time `solar` must be below **0.5 % of the series maximum**, not exactly zero.
The justification is quantified immediately after the check.

In [ ]:
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]
solar_tol = 0.005 * ts["solar"].max()
night = ts["hour"].isin(NIGHT_HOURS)

checks = {
    "wind_off < 0": ts["wind_off"] < 0,
    "wind_on < 0": ts["wind_on"] < 0,
    "solar < 0": ts["solar"] < 0,
    "grid_load <= 0": ts["grid_load"] <= 0,
    f"night solar > {solar_tol:,.1f} MWh": night & (ts["solar"] > solar_tol),
}

print(f"solar max {ts['solar'].max():,.1f} MWh -> tolerance {solar_tol:,.1f} MWh")
print(f"night hours defined as {NIGHT_HOURS}\n")

for name, mask in checks.items():
    print(f"{name:<38} {mask.sum():>6} violations")
    if mask.any():
        display(ts.loc[mask, SERIES])  # list every real violation by timestamp

print(
    f"\nnight solar: max {ts.loc[night, 'solar'].max():,.2f} MWh, "
    f"{(ts.loc[night, 'solar'] == 0).sum():,} of {night.sum():,} night hours are exactly zero"
)

**No violations.** Generation is never negative, `grid_load` is always strictly positive, and no
night hour comes anywhere near the tolerance — the largest night-time solar value in the record
is about **147 MWh** against a **290 MWh** threshold, i.e. half of it.

The tolerance earns its place: only about 400 of the ~10 300 night hours are *exactly* zero, so a
strict `solar == 0` night test would report roughly **9 900 "violations"** and invite a
zero-inflation claim that the data does not support. Those few MWh are measurement and reporting
noise, not generation.

### 5.6 · The residual load identity

The project's target is `residual_load`. Before building anything on it, confirm empirically what
it *is* — and report the **distribution** of the difference rather than a yes/no, so that
two-decimal CSV rounding is not mistaken for a discrepancy.

In [ ]:
diff_actual = ts["residual_load"] - (ts["grid_load"] - ts["renewables"])
diff_fc = ts["fc_res"] - (ts["fc_grid_load"] - ts["fc_gen_wind_solar"])

report = (
    pd.DataFrame({"actual": diff_actual.abs(), "forecast": diff_fc.abs()})
    .describe(percentiles=[0.5, 0.95, 0.99])
    .T.round(4)
)
report["max_abs"] = [diff_actual.abs().max(), diff_fc.abs().max()]
report["n_above_0.5_MWh"] = [(diff_actual.abs() > 0.5).sum(), (diff_fc.abs() > 0.5).sum()]
display(report)

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.hist(diff_actual, bins=41, color="C0")
ax.set_title(
    "residual_load  minus  (grid_load - wind_on - wind_off - solar)", fontsize=13, pad=10
)
ax.set_xlabel("difference (MWh)")
ax.set_ylabel("hours", color="grey")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

The difference is bounded by **0.02 MWh** for the actuals and **0.01 MWh** for the forecasts, with
not a single hour above 0.5 MWh, and the histogram is a spike on a grid of two-decimal steps. On
a typical 50 000 MWh hour that residue is about 4 × 10⁻⁷ of the value — it is the CSV's
two-decimal format, nothing more.

So, stated plainly for the rest of the project to cite rather than re-open as an assumption:

> `residual_load` **is** `grid_load − (wind_on + wind_off + solar)`, exactly, and
> `fc_gen_wind_solar`'s actual counterpart **is** the summed wind and solar generation.

Two things follow. First, any near-perfect correlation between `grid_load`, the renewables and
`residual_load` in §9 is arithmetic, not a discovery, and must not be presented as a feature
finding. Second, modelling `residual_load` directly and modelling load-minus-renewables are the
same problem, so the choice between them is about error structure, not about definitions.

---

## 6 · Conventions and correctness

**This section is a contract, not a private choice.** The week convention, the bin label side,
the ISO grouping rule, the reporting units, the season definition and the descriptive-slice
policy fixed here are inherited by [`specs/02-Deep-EDA.md`](../specs/02-Deep-EDA.md) and by the
modeling work, which cite them rather than re-deriving them.

### 6.1 · The week convention: ISO, Monday start, Sunday end

In pandas, `"W"` is an alias for `"W-SUN"` — weeks *ending* Sunday, which is exactly ISO
Monday-start weeks. The alias reads as though it meant the opposite, so it is worth asserting
rather than trusting. But it has to be the *right* assertion: `to_period("W").start_time` is a
Monday for every bin **by construction**, including partial ones, so testing that would pass
without testing anything. The meaningful check is on the first *data* timestamp that falls into
each complete bin.

In [ ]:
weekly_bins = ts.index.to_period("W")
first_ts_in_bin = pd.Series(ts.index, index=weekly_bins).groupby(level=0).min()

complete_weeks = _complete_periods(ts.index, "W")
checked = first_ts_in_bin.loc[complete_weeks]

assert (checked.dt.dayofweek == 0).all() and (checked.dt.hour == 0).all(), \
    "a complete weekly bin does not open on Monday 00:00"
print(f"{len(checked):,} complete weekly bins, every one opening Monday 00:00")

# The counter-example that shows the test is not vacuous: the excluded first bin.
excluded_bin, excluded_ts = first_ts_in_bin.index[0], first_ts_in_bin.iloc[0]
print(
    f"excluded first bin {excluded_bin} opens {excluded_ts:%Y-%m-%d}, a {excluded_ts:%A} "
    "-- the record starts mid-week, which is why that bin is dropped rather than trusted"
)

244 complete weekly bins, every one opening Monday 00:00. The first bin is deliberately excluded
because the record opens on **Saturday 2022-01-01** — and printing that is what gives the
assertion teeth: a test that only ever saw Mondays would pass whether or not the convention held.

### 6.2 · Which edge of the week the label refers to

The two routes to a weekly aggregate label the same week with **different dates**:

In [ ]:
first_week = ts.index.to_period("W")[0]
print(f"the week bin {first_week}")
print(f"  period_mean    labels it {first_week.start_time:%Y-%m-%d} ({first_week.start_time:%A}) -- period START")
print(f"  resample('W')  labels it {first_week.end_time:%Y-%m-%d} ({first_week.end_time:%A}) -- RIGHT EDGE")

**Convention for this project: weekly x values are the Monday (the period start),** because every
weekly aggregate here goes through `period_mean`. Weekly axes are labelled accordingly, and §6.3
relabels `.resample()`'s output before comparing indexes for exactly this reason. An unlabelled
weekly axis is ambiguous by a six-day offset.

### 6.3 · Is `period_mean` correct?

`period_mean` is a plain calendar-period mean plus **one** extra rule. Before the rest of the
notebook leans on it, test it once against a plain `.resample()`.

These are the **only two `.resample()` calls in this notebook**. Everywhere else, weekly and
monthly aggregation goes through `period_mean` or `period_energy`. Note the `"ME"` spelling for
the monthly case: this project runs pandas 3, where `"M"` has been removed as an offset alias and
`.resample("M")` raises `ValueError`. `to_period("M")` is unaffected, so `period_mean(s, "M")`
itself still works — only the comparison side needs the new spelling.

In [ ]:
for freq, resample_alias in [("W", "W"), ("M", "ME")]:
    pm = period_mean(ts["grid_load"], freq)
    res = ts["grid_load"].resample(resample_alias).mean()

    # 1. Relabel before comparing anything: period_mean labels the start, resample the right edge.
    res_relabelled = res.copy()
    res_relabelled.index = res.index.to_period(freq).start_time

    # 2. On the periods both produce, the values must agree.
    shared = pm.index.intersection(res_relabelled.index)
    assert np.allclose(pm.loc[shared], res_relabelled.loc[shared])

    # 3. Exactly the INCOMPLETE periods are dropped -- expected set computed, not hard-coded.
    #    Restated from first principles: only the first and last period CAN be partial, so test
    #    each of those two against the data bounds.
    edges = dict.fromkeys([ts.index.min().to_period(freq), ts.index.max().to_period(freq)])
    expected_dropped = pd.PeriodIndex(
        [
            p for p in edges
            if p.start_time < ts.index.min() or p.end_time > ts.index.max() + pd.Timedelta("1h")
        ],
        freq=freq,
    )
    actual_dropped = res_relabelled.index.difference(pm.index)
    assert set(expected_dropped.start_time) == set(actual_dropped)

    print(f"{freq}: period_mean keeps {len(pm)}, resample produces {len(res)}, "
          f"agreeing on all {len(shared)} shared periods")

    # 4. Print each dropped edge next to a complete neighbour, so the artefact is visible in MW.
    for p in expected_dropped:
        neighbour = p + 1 if p.start_time < ts.index.min() else p - 1
        dropped_val = res_relabelled.loc[p.start_time]
        neighbour_val = res_relabelled.loc[neighbour.start_time]
        print(f"   dropped {p}: {dropped_val:>10,.0f} MW   vs neighbour {neighbour}: "
              f"{neighbour_val:>10,.0f} MW   ({dropped_val / neighbour_val - 1:+.1%})")
    print()

`period_mean` is a plain calendar-period mean plus one rule: **drop periods the data does not
fully cover**. The test above confirms both halves of that claim — the arithmetic is identical to
`.resample().mean()` on every period the two share, and the periods it drops are exactly the
incomplete ones, with the expected set computed from the data bounds rather than hard-coded.

**The rule bites asymmetrically here**, which is why "drop the first and last period" is the wrong
way to describe it. The record starts at exactly `2022-01-01 00:00`:

- **Weekly** — both edges go. 2022-01-01 is a Saturday, so the opening week is partial, and the
  record ends on a Wednesday, so the closing week is too. 244 of 246.
- **Monthly** — only the trailing edge goes. January 2022 is *complete*, because the data starts
  precisely on the month boundary. 56 of 57.

The printed values show the size of the artefact being avoided, and they correct a natural
assumption: the partial edges are **not** always dips. The opening week reads ~23 % *low* — it is
two public holidays and a weekend, nothing else. The closing week reads ~8 % *high*, because it
covers Monday to Wednesday only, three working days with no weekend to pull the mean down. The
distortion is signed by whichever days the partial period happens to contain, so "fake edge dip"
understates the problem: the edge can lie in either direction.

This test runs **once**, here, as a correctness check. It is not repeated for every later
aggregation.

### 6.4 · Does month length contaminate our comparisons?

`period_energy` (§2) exists so that month length cannot silently distort month-over-month
comparison. It returns two units — **MWh per day** (period sum ÷ calendar days) and **average
MW** (period sum ÷ hours *actually present*) — and drops incomplete periods on the same rule as
`period_mean`.

The table below puts four aggregation variants side by side for `grid_load`, with both
denominators exposed. It is shown with `drop_incomplete=False` so that the period the rule
discards is visible rather than merely described.

In [ ]:
pe = period_energy(ts["grid_load"], "M", drop_incomplete=False)
hourly_mean = ts["grid_load"].groupby(ts.index.to_period("M")).mean()

tab = pe.assign(
    hourly_mean=hourly_mean.to_numpy(),
    naive=pe["mwh_per_day"] / 24,  # the naive denominator: sum / (24 * days)
)
tab["naive_err_%"] = (100 * (tab["naive"] / tab["avg_mw"] - 1)).round(4)
tab["complete"] = tab.index.isin(period_energy(ts["grid_load"], "M").index)
tab = tab[["hourly_mean", "avg_mw", "mwh_per_day", "naive", "days", "hours", "naive_err_%", "complete"]]

# The negative result, stated as an assertion: not "close", identical.
assert (tab["hourly_mean"] - tab["avg_mw"]).abs().max() == 0.0

print("first three months:")
display(tab.head(3).round(2))

print("every month where hours != 24 x days -- the only rows where the naive denominator lies:")
display(tab[tab["hours"] != 24 * tab["days"]].round(2))

In [ ]:
sums = ts["grid_load"].groupby(ts.index.to_period("M")).sum()
jan, feb = pd.Period("2022-01", "M"), pd.Period("2022-02", "M")

print("February 2022 vs January 2022, same data, two aggregations:")
print(f"  raw monthly sum : {100 * (sums[feb] / sums[jan] - 1):+.1f} %   "
      f"({sums[jan]:,.0f} -> {sums[feb]:,.0f} MWh)")
print(f"  MWh per day     : "
      f"{100 * (pe['mwh_per_day'][feb.start_time] / pe['mwh_per_day'][jan.start_time] - 1):+.1f} %")
print(f"  average MW      : "
      f"{100 * (pe['avg_mw'][feb.start_time] / pe['avg_mw'][jan.start_time] - 1):+.1f} %")

Four conclusions, one of which is a negative result worth writing down:

1. **The hourly mean and average MW are the same number, always.** Not approximately — the
   assertion above is an exact `== 0.0` over all 57 months. Both divide by hours present, and
   since a value in MWh over a one-hour interval *is* the average MW, the two are the same
   quantity by construction.
2. **MWh per day is that number rescaled by days-per-month.** For *means*, month length cancels
   completely.
3. **The distortion the helper guards against appears only under the naive
   `sum / (24 × days)` denominator, and only where hours are missing.** The five March months
   hold 743 hours instead of 744 and read about **0.13 % low**. September 2026 holds 216 hours
   against 30 days and reads ~70 % low — but the edge rule drops it anyway, so it never reaches
   a plot.
4. **It is when we aggregate *sums* that ignoring month length distorts things**, and there the
   effect is large enough to invent a trend that is not there: February 2022 looks like an
   **8.6 % collapse** against January on raw sums, and is a **1.2 % rise** on MWh per day. The
   entire apparent drop is three fewer days.

So the mean-based views used throughout this notebook were fine all along. That is the honest
finding, and it is worth recording precisely so nobody re-opens the question later.

### 6.5 · Project conventions fixed here

**Reporting units.** **Average MW** for load and generation *levels*; **MWh per day** only where
the quantity is genuinely an energy volume. Every plot states which one it shows. This is why
`ylabel` is a required argument in the `style_timeseries` and `seasonal_plot` copies in §2 — the
originals default it to `"(MWh)"`, which would let a plot silently mislabel its own units.

**Seasons.** Meteorological seasons, with **December assigned to the following year's winter**:

In [ ]:
display(
    ts.groupby(["season_year", "season"], observed=True).size().unstack(fill_value=0)
)
print("season_year = year + (month == 12)")

winter = Dec/Jan/Feb, spring = Mar/Apr/May, summer = Jun/Jul/Aug, autumn = Sep/Oct/Nov, with
`season_year = year + (month == 12)`.

The December rule is not cosmetic. Under the alternative — December stays in its own calendar
year — winter 2022 would be January + February + December 2022, three months that never occurred
consecutively, and it would *look complete* when it is not. Under our rule, winter 2022 is
visibly Jan + Feb only — the table above shows **1 416 hours against ~2 160** for a full winter —
which is the honest representation of a record that starts on 1 January. Autumn 2026 is flagged
by the same mechanism at 216 hours. And because the data ends before December 2026, no orphan
2027 winter group arises.

**Descriptive slices.** Showing "the tail hours" requires selecting them, which looks like
thresholding but is not. The policy, stated here once so later sections do not collide with the
no-threshold rule:

> Rank-based slices — top/bottom 1 % by rank, largest-N by magnitude, longest run — are used
> **for description only**. No boolean column is created, nothing is persisted, and no slice is
> presented as a risk definition. Slices are computed inside the plotting cell and never added to
> `ts`.

The line is crisp: it becomes a threshold the moment a boolean column is created or a slice is
named a risk case. Defining the risk flag is the modeling spec's job, and §11's column assertion
is the mechanical check that this notebook did not quietly do it first.

---

## 7 · Univariate description

All eight series, described one at a time: summary statistics, the full-period level, and the
shape of the distribution. The three SMARD `fc_*` forecast columns are treated here as **ordinary
series** — how well they predict their actuals is a forecast-benchmark question, and that belongs
to [spec 02](../specs/02-Deep-EDA.md), not here.

### 7.1 · Summary statistics

In [ ]:
display(
    ts[SERIES]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .T.drop(columns="count")
    .round(1)
)

Four things stand out.

**The renewables are hugely more variable than load.** `wind_on` has a standard deviation
(~9 700) almost as large as its mean (~12 400), and `solar`'s std exceeds its mean outright. By
contrast `grid_load` sits at ~53 300 ± 9 300 — it moves within a band, while generation swings
between nothing and tens of thousands of MW.

**`solar` is strongly zero-inflated by construction.** Its 25th percentile is ~6 MWh and its
median ~312 MWh, against a maximum of 58 056: half the hours in the year are night or near-night.
This is a property of the physical process, not a data defect (§5.5), but it makes the mean a
poor summary of solar and it is why the hour-of-day structure in §8 matters so much.

**Both residual load series reach below zero** — `residual_load` to −15 562 and `fc_res` to
−29 003. The percentiles locate the negative tail quite precisely: the **1st percentile is already
negative** (−3 501 and −3 487) while the 5th is solidly positive (+5 110 and +5 396), so somewhere
between 1 % and 5 % of hours sit below zero. §10 pins that down to 2.08 % and describes it
properly.

**The forecasts are not centred on their actuals.** `fc_grid_load` averages ~53 603 against
`grid_load`'s ~53 336, and `fc_res` averages ~30 628 against ~30 240. A few hundred MW of
systematic offset is visible even at this resolution — quantifying it is spec 02's job, but it is
worth noticing that the SMARD day-ahead forecasts are not unbiased.

### 7.2 · Level over the full period

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 17))

for ax, col in zip(axes.flat, SERIES):
    weekly = period_mean(ts[col], "W")
    ax.plot(weekly.index, weekly.to_numpy(), color="C0", linewidth=1.1)
    ax.axhline(0, color="0.6", linewidth=0.8, zorder=0)
    style_timeseries(ax, col, "average MW")
    ax.set_xlabel("week (labelled by its Monday)", color="grey", fontsize=9)

fig.suptitle(
    "All eight series, weekly means (incomplete edge weeks dropped)", fontsize=16, y=0.997
)
plt.tight_layout()
plt.show()

Weekly means, via `period_mean`, so the partial opening and closing weeks of §6.3 are not
plotted. Read at this resolution:

- **`grid_load`** has a clean annual cycle — winter high, summer low — riding on a visible
  **downward level shift after 2022**. §8.5 tests that properly rather than eyeballing it.
- **`solar`** is the most regular series in the set: a near-sinusoidal annual cycle whose summer
  peaks grow year on year, consistent with continued capacity build-out.
- **`wind_on` and `wind_off`** are the opposite — winter-weighted but dominated by weather, so
  the week-to-week scatter is large and no annual shape is as crisp as solar's.
- **`residual_load`** inherits load's annual cycle with the renewables subtracted out, which
  deepens its summer troughs over time. The horizontal zero line shows that weekly *means* never
  approach zero; the negative hours of §10 are an hourly phenomenon that weekly aggregation hides
  completely.
- The three **`fc_*`** series track their actuals closely enough to be visually indistinguishable
  at this scale, which is exactly why a proper error metric (spec 02) is needed to say anything
  useful about them.

### 7.3 · Distributions

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 13))

for ax, col in zip(axes.flat, SERIES):
    ax.hist(ts[col], bins=80, color="C0")
    ax.axvline(0, color="C3", linewidth=1.0)
    ax.set_title(col, fontsize=12, pad=8)
    ax.set_xlabel("MWh per hour")
    ax.set_ylabel("hours", color="grey")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

fig.suptitle("Hourly distribution of each series (red line = zero)", fontsize=16, y=0.999)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))

ax.boxplot(
    [ts[col] for col in SERIES],
    tick_labels=SERIES,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.25},
    medianprops={"color": "C1"},
)
ax.axhline(0, color="C3", linewidth=1.0)
ax.set_title("Spread and tails of each series", fontsize=15, pad=12)
ax.set_ylabel("MWh per hour", color="grey")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="x", rotation=30)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
plt.tight_layout()
plt.show()

The histograms and the boxplot say complementary things.

**Shapes split into three families.** `grid_load`, `fc_grid_load`, `residual_load` and `fc_res`
are broad and roughly unimodal — these are the series a model can reasonably target. `wind_on`,
`wind_off` and `fc_gen_wind_solar` are heavily right-skewed, piling up at low values with a long
thin tail of storm hours. `solar` is a different object again: a spike at the bottom of the range
holding more than half the record, then a wide shoulder. Averaging solar across all hours
describes no actual hour.

**The tails are asymmetric in a way that matters for this project.** The boxplot shows both
residual load series carrying whiskers and outliers in *both* directions — the high tail reaching
past 70 000 MWh and the negative tail crossing zero — whereas the generation series can only have
one. Those two directions are the two risk cases the project is built around (tight margins
above, renewable oversupply below), and §10 describes both.

**A boxplot is the wrong tool for `solar` and an informative one for `residual_load`.** Solar's
box is compressed against zero with thousands of "outliers" that are simply daylight hours — the
interquartile range describes the night, not the process. For `residual_load` the same plot is
genuinely diagnostic, because its distribution is close enough to symmetric that the whiskers
mean what they usually mean.

Note that no log scale appears anywhere above: `residual_load` and `fc_res` take negative values,
so log transforms are unusable for them, and applying one to the other series would have made the
panels mutually incomparable.